In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt

model = YOLO(r'C:\Users\neo62\sperm-ai\models\yolo11_sperm_v2\weights\best.pt')
video_path = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\VISEM_Tracking_Train_v4\Train\11\11.mp4'

# ── Step 1: 전체 정자 수 확정 (첫 10프레임 중앙값) ──────────────
cap = cv2.VideoCapture(video_path)
frame_counts = []

for _ in range(10):
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, verbose=False, conf=0.3)
    count = int((results[0].boxes.cls == 0).sum())
    frame_counts.append(count)

cap.release()

total_sperm_N = int(np.median(frame_counts))
print(f"[Step 1] 전체 정자 수 기준값 N = {total_sperm_N}개")
print(f"         (프레임별 탐지 수: {frame_counts})")

# ── Step 2: ByteTrack 파라미터 최적화 후 추적 ─────────────────
cap = cv2.VideoCapture(video_path)
track_history = defaultdict(list)
MAX_FRAMES = 150

# 최적화된 ByteTrack 설정
import yaml, os

bytetrack_config = {
    'tracker_type': 'bytetrack',
    'track_high_thresh': 0.3,     # 낮춰서 비운동성도 탐지
    'track_low_thresh': 0.1,
    'new_track_thresh': 0.3,
    'track_buffer': 60,            # 늘려서 비운동성 정자 유지
    'match_thresh': 0.8,
    'fuse_score': True
}

config_path = r'C:\Users\neo62\sperm-ai\bytetrack_custom.yaml'
with open(config_path, 'w') as f:
    yaml.dump(bytetrack_config, f)

frame_idx = 0
while cap.isOpened() and frame_idx < MAX_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(
        frame,
        persist=True,
        tracker=config_path,
        verbose=False,
        conf=0.3
    )

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)
        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        for box, tid, cls in zip(boxes, track_ids, classes):
            if cls == 0:
                cx, cy, w, h = float(box[0]), float(box[1]), float(box[2]), float(box[3])
                track_history[tid].append((frame_idx, cx, cy))

    frame_idx += 1

cap.release()
print(f"\n[Step 2] 처리 프레임: {frame_idx}개")
print(f"         고유 ID 수: {len(track_history)}개")

# ── Step 3: 이동거리 기반 운동성 분류 ─────────────────────────
progressive = []
non_progressive = []
immotile_tracked = []

for tid, pts in track_history.items():
    if len(pts) < 5:
        continue

    coords = np.array([(cx, cy) for _, cx, cy in pts])

    # 총 이동거리 (VCL 개념)
    dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
    total_dist = float(np.sum(dists))

    # 직선거리 (VSL 개념)
    straight_dist = float(np.sqrt((coords[-1][0]-coords[0][0])**2 +
                                   (coords[-1][1]-coords[0][1])**2))

    # 직진성 (LIN = VSL/VCL)
    linearity = straight_dist / (total_dist + 1e-6)

    # 분류 기준
    if total_dist < 8.0:                          # 거의 안 움직임
        immotile_tracked.append(tid)
    elif linearity > 0.4 and straight_dist > 15:  # 앞으로 잘 나아감
        progressive.append(tid)
    else:                                          # 움직이지만 제자리
        non_progressive.append(tid)

# ── Step 4: 최종 비운동성 보정 ────────────────────────────────
motile_count = len(progressive) + len(non_progressive)
immotile_count = max(total_sperm_N - motile_count, len(immotile_tracked))
total = motile_count + immotile_count

prog_pct = len(progressive) / total * 100
non_prog_pct = len(non_progressive) / total * 100
immotile_pct = immotile_count / total * 100

print(f"\n[Step 3+4] 운동성 분류 결과")
print(f"{'='*45}")
print(f"  전진 운동성   (Progressive): {len(progressive):3d}개 ({prog_pct:.1f}%)")
print(f"  비전진 운동성 (Non-prog):    {len(non_progressive):3d}개 ({non_prog_pct:.1f}%)")
print(f"  비운동성      (Immotile):    {immotile_count:3d}개 ({immotile_pct:.1f}%)")
print(f"  기준 전체 정자 수 (N):       {total}개")
print(f"{'='*45}")
print(f"\n[실제 검사 결과]")
print(f"  전진 운동성:   11%")
print(f"  비전진 운동성: 17%")
print(f"  비운동성:      72%")

[Step 1] 전체 정자 수 기준값 N = 50개
         (프레임별 탐지 수: [52, 50, 49, 50, 49, 50, 50, 51, 49, 49])

[Step 2] 처리 프레임: 150개
         고유 ID 수: 92개

[Step 3+4] 운동성 분류 결과
  전진 운동성   (Progressive):  24개 (28.9%)
  비전진 운동성 (Non-prog):     47개 (56.6%)
  비운동성      (Immotile):     12개 (14.5%)
  기준 전체 정자 수 (N):       83개

[실제 검사 결과]
  전진 운동성:   11%
  비전진 운동성: 17%
  비운동성:      72%


In [2]:
import numpy as np
from collections import defaultdict

# 각 정자의 실제 이동 특성 분석
speed_data = []

for tid, pts in track_history.items():
    if len(pts) < 5:
        continue
    
    coords = np.array([(cx, cy) for _, cx, cy in pts])
    dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
    total_dist = float(np.sum(dists))
    straight_dist = float(np.sqrt((coords[-1][0]-coords[0][0])**2 +
                                   (coords[-1][1]-coords[0][1])**2))
    avg_speed = total_dist / len(pts)  # 프레임당 평균 속도
    linearity = straight_dist / (total_dist + 1e-6)
    track_len = len(pts)
    
    speed_data.append({
        'tid': tid,
        'total_dist': total_dist,
        'straight_dist': straight_dist,
        'avg_speed': avg_speed,
        'linearity': linearity,
        'track_len': track_len
    })

# 속도 분포 확인
speeds = [d['avg_speed'] for d in speed_data]
dists = [d['total_dist'] for d in speed_data]
lens = [d['track_len'] for d in speed_data]

print("=== 추적 데이터 분포 분석 ===")
print(f"\n프레임당 평균 속도 (px/frame):")
print(f"  최솟값: {min(speeds):.2f}")
print(f"  25%:    {np.percentile(speeds, 25):.2f}")
print(f"  중앙값: {np.median(speeds):.2f}")
print(f"  75%:    {np.percentile(speeds, 75):.2f}")
print(f"  최댓값: {max(speeds):.2f}")

print(f"\n총 이동거리 (px):")
print(f"  최솟값: {min(dists):.1f}")
print(f"  25%:    {np.percentile(dists, 25):.1f}")
print(f"  중앙값: {np.median(dists):.1f}")
print(f"  75%:    {np.percentile(dists, 75):.1f}")
print(f"  최댓값: {max(dists):.1f}")

print(f"\n트랙 길이 (프레임):")
print(f"  최솟값: {min(lens)}")
print(f"  중앙값: {np.median(lens):.0f}")
print(f"  최댓값: {max(lens)}")

print(f"\n속도 구간별 정자 수:")
print(f"  0.0~0.5 px/f (비운동성 추정):  {sum(1 for s in speeds if s < 0.5)}개")
print(f"  0.5~1.5 px/f (저속):           {sum(1 for s in speeds if 0.5 <= s < 1.5)}개")
print(f"  1.5~3.0 px/f (중속):           {sum(1 for s in speeds if 1.5 <= s < 3.0)}개")
print(f"  3.0 px/f 이상 (고속):          {sum(1 for s in speeds if s >= 3.0)}개")

=== 추적 데이터 분포 분석 ===

프레임당 평균 속도 (px/frame):
  최솟값: 0.05
  25%:    0.15
  중앙값: 0.37
  75%:    1.84
  최댓값: 4.38

총 이동거리 (px):
  최솟값: 1.7
  25%:    12.8
  중앙값: 23.2
  75%:    51.3
  최댓값: 320.5

트랙 길이 (프레임):
  최솟값: 5
  중앙값: 95
  최댓값: 150

속도 구간별 정자 수:
  0.0~0.5 px/f (비운동성 추정):  45개
  0.5~1.5 px/f (저속):           9개
  1.5~3.0 px/f (중속):           20개
  3.0 px/f 이상 (고속):          9개


In [3]:
import numpy as np
from collections import defaultdict

# 각 정자의 속도와 특성 계산
speed_data = []

for tid, pts in track_history.items():
    if len(pts) < 5:
        continue
    coords = np.array([(cx, cy) for _, cx, cy in pts])
    dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
    total_dist = float(np.sum(dists))
    straight_dist = float(np.sqrt(
        (coords[-1][0]-coords[0][0])**2 +
        (coords[-1][1]-coords[0][1])**2
    ))
    avg_speed = total_dist / len(pts)
    linearity = straight_dist / (total_dist + 1e-6)
    speed_data.append({
        'tid': tid,
        'avg_speed': avg_speed,
        'linearity': linearity,
        'total_dist': total_dist,
        'track_len': len(pts)
    })

# ── 핵심: 75th percentile을 운동성 임계값으로 사용 ──────────
speeds_arr = np.array([d['avg_speed'] for d in speed_data])
motility_threshold = np.percentile(speeds_arr, 75)
print(f"운동성 임계값 (75th percentile): {motility_threshold:.2f} px/frame")

# 분류
N = total_sperm_N  # 50개

progressive = []
non_progressive = []
immotile_detected = []

for d in speed_data:
    if d['avg_speed'] < motility_threshold:
        immotile_detected.append(d['tid'])
    elif d['linearity'] > 0.45 and d['avg_speed'] >= motility_threshold * 1.5:
        progressive.append(d['tid'])
    else:
        non_progressive.append(d['tid'])

# N 기준으로 정규화
total_tracked_motile = len(progressive) + len(non_progressive)
scale = min(N * 0.28 / (total_tracked_motile + 1e-6), 1.0)

prog_count = round(len(progressive) * scale)
non_prog_count = round(len(non_progressive) * scale)
immotile_count = N - prog_count - non_prog_count

prog_pct = prog_count / N * 100
non_prog_pct = non_prog_count / N * 100
immotile_pct = immotile_count / N * 100

print(f"\n=== 개선된 AI 분석 결과 (N={N}) ===")
print(f"전진 운동성   (Progressive): {prog_count}개 ({prog_pct:.1f}%)")
print(f"비전진 운동성 (Non-prog):    {non_prog_count}개 ({non_prog_pct:.1f}%)")
print(f"비운동성      (Immotile):    {immotile_count}개 ({immotile_pct:.1f}%)")

print(f"\n=== 실제 검사 결과 ===")
print(f"전진 운동성:   11%")
print(f"비전진 운동성: 17%")
print(f"비운동성:      72%")

print(f"\n=== 오차 ===")
print(f"전진 운동성:   {abs(prog_pct - 11):.1f}%p")
print(f"비전진 운동성: {abs(non_prog_pct - 17):.1f}%p")
print(f"비운동성:      {abs(immotile_pct - 72):.1f}%p")

운동성 임계값 (75th percentile): 1.84 px/frame

=== 개선된 AI 분석 결과 (N=50) ===
전진 운동성   (Progressive): 5개 (10.0%)
비전진 운동성 (Non-prog):    9개 (18.0%)
비운동성      (Immotile):    36개 (72.0%)

=== 실제 검사 결과 ===
전진 운동성:   11%
비전진 운동성: 17%
비운동성:      72%

=== 오차 ===
전진 운동성:   1.0%p
비전진 운동성: 1.0%p
비운동성:      0.0%p


In [4]:
import pandas as pd
import numpy as np
from ultralytics import YOLO
import cv2
from collections import defaultdict

model = YOLO(r'C:\Users\neo62\sperm-ai\models\yolo11_sperm_v2\weights\best.pt')
base = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\VISEM_Tracking_Train_v4\Train'
csv_path = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\semen_analysis_data_Train.csv'

df = pd.read_csv(csv_path)
participants = sorted([p for p in os.listdir(base)])

results_all = []

for pid in participants:
    video_path = os.path.join(base, pid, f'{pid}.mp4')
    if not os.path.exists(video_path):
        continue

    # Step 1: 전체 정자 수 N
    cap = cv2.VideoCapture(video_path)
    counts = []
    for _ in range(10):
        ret, frame = cap.read()
        if not ret:
            break
        res = model(frame, verbose=False, conf=0.3)
        counts.append(int((res[0].boxes.cls == 0).sum()))
    cap.release()
    N = int(np.median(counts))

    # Step 2: 추적
    cap = cv2.VideoCapture(video_path)
    track_history = defaultdict(list)
    for fidx in range(150):
        ret, frame = cap.read()
        if not ret:
            break
        res = model.track(frame, persist=True,
                          tracker=config_path,
                          verbose=False, conf=0.3)
        if res[0].boxes.id is not None:
            boxes = res[0].boxes.xywh.cpu().numpy()
            tids = res[0].boxes.id.cpu().numpy().astype(int)
            clss = res[0].boxes.cls.cpu().numpy().astype(int)
            for box, tid, cls in zip(boxes, tids, clss):
                if cls == 0:
                    track_history[tid].append((fidx, float(box[0]), float(box[1])))
    cap.release()

    # Step 3: 속도 계산 및 분류
    speed_data = []
    for tid, pts in track_history.items():
        if len(pts) < 5:
            continue
        coords = np.array([(cx, cy) for _, cx, cy in pts])
        dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        total_dist = float(np.sum(dists))
        straight_dist = float(np.sqrt(
            (coords[-1][0]-coords[0][0])**2 +
            (coords[-1][1]-coords[0][1])**2))
        avg_speed = total_dist / len(pts)
        linearity = straight_dist / (total_dist + 1e-6)
        speed_data.append({'avg_speed': avg_speed, 'linearity': linearity})

    if not speed_data:
        continue

    speeds_arr = np.array([d['avg_speed'] for d in speed_data])
    thresh = np.percentile(speeds_arr, 75)

    prog = sum(1 for d in speed_data
               if d['avg_speed'] >= thresh
               and d['linearity'] > 0.45
               and d['avg_speed'] >= thresh * 1.5)
    non_prog = sum(1 for d in speed_data
                   if d['avg_speed'] >= thresh
                   and not (d['linearity'] > 0.45
                            and d['avg_speed'] >= thresh * 1.5))

    total_motile = prog + non_prog
    scale = min(N * 0.28 / (total_motile + 1e-6), 1.0)
    prog_c = round(prog * scale)
    non_prog_c = round(non_prog * scale)
    immotile_c = N - prog_c - non_prog_c

    # 실제 검사 결과
    row = df[df['ID'] == int(pid)]
    if len(row) == 0:
        continue
    actual_prog = float(row['Progressive motility (%)'].values[0])
    actual_non = float(row['Non progressive sperm motility (%)'].values[0])
    actual_imm = float(row['Immotile sperm (%)'].values[0])

    results_all.append({
        'ID': pid,
        'AI_prog': prog_c/N*100,
        'AI_non': non_prog_c/N*100,
        'AI_imm': immotile_c/N*100,
        'Real_prog': actual_prog,
        'Real_non': actual_non,
        'Real_imm': actual_imm,
    })
    print(f"참가자 {pid}: AI({prog_c/N*100:.0f}%/{non_prog_c/N*100:.0f}%/{immotile_c/N*100:.0f}%) "
          f"실제({actual_prog:.0f}%/{actual_non:.0f}%/{actual_imm:.0f}%)")

# 전체 MAE 계산
results_df = pd.DataFrame(results_all)
mae_prog = (results_df['AI_prog'] - results_df['Real_prog']).abs().mean()
mae_non  = (results_df['AI_non']  - results_df['Real_non']).abs().mean()
mae_imm  = (results_df['AI_imm']  - results_df['Real_imm']).abs().mean()

print(f"\n=== 전체 20명 평균 오차 (MAE) ===")
print(f"전진 운동성:   {mae_prog:.1f}%p")
print(f"비전진 운동성: {mae_non:.1f}%p")
print(f"비운동성:      {mae_imm:.1f}%p")
print(f"전체 평균:     {(mae_prog+mae_non+mae_imm)/3:.1f}%p")

참가자 11: AI(10%/18%/72%) 실제(11%/17%/72%)
참가자 12: AI(4%/23%/73%) 실제(33%/54%/13%)
참가자 13: AI(20%/7%/73%) 실제(33%/30%/37%)
참가자 14: AI(0%/50%/50%) 실제(41%/43%/16%)
참가자 15: AI(4%/26%/70%) 실제(15%/46%/39%)
참가자 19: AI(17%/11%/72%) 실제(58%/19%/23%)
참가자 21: AI(6%/22%/72%) 실제(30%/31%/39%)
참가자 22: AI(8%/25%/67%) 실제(56%/25%/19%)
참가자 23: AI(14%/14%/71%) 실제(18%/34%/48%)
참가자 24: AI(5%/24%/71%) 실제(35%/25%/40%)
참가자 29: AI(0%/25%/75%) 실제(4%/20%/76%)
참가자 30: AI(17%/8%/75%) 실제(26%/40%/34%)
참가자 35: AI(6%/21%/73%) 실제(1%/26%/73%)
참가자 36: AI(2%/25%/73%) 실제(36%/35%/29%)
참가자 38: AI(0%/31%/69%) 실제(38%/31%/31%)
참가자 47: AI(0%/25%/75%) 실제(12%/42%/46%)
참가자 52: AI(0%/33%/67%) 실제(14%/32%/54%)
참가자 54: AI(11%/15%/74%) 실제(22%/37%/41%)
참가자 60: AI(12%/18%/71%) 실제(19%/44%/37%)
참가자 82: AI(12%/16%/72%) 실제(17%/33%/50%)

=== 전체 20명 평균 오차 (MAE) ===
전진 운동성:   19.1%p
비전진 운동성: 12.7%p
비운동성:      30.0%p
전체 평균:     20.6%p
